# Scraper DB Inspection

Quick notebook to inspect `data/scraper.db` with FastLite.

In [ ]:
from pathlib import Path
from fastlite import database

candidates = [
    Path("data/scraper.db"),
    Path("../data/scraper.db"),
    Path("../../data/scraper.db"),
    Path("scraper.db"),
]

db_path = next((p.resolve() for p in candidates if p.exists()), None)
if db_path is None:
    raise FileNotFoundError("Could not find scraper.db. Checked: " + ", ".join(str(p) for p in candidates))

db = database(str(db_path))
print(f"Connected to: {db_path}")

Connected to: C:\python\hierag\data\scraper.db


In [2]:
try:
    import pandas as pd
except ImportError:
    pd = None


def to_table(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)


tables = list(db.q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"))
to_table(tables)

,name
0,cache_entries
1,chats
2,chunks
3,discovered_urls
4,embeddings
5,extracts
6,feedback
7,messages
8,pages
9,pdfs


In [3]:
TABLE_NAME = "discovered_urls"  # change this if you want to inspect another table

columns = list(db.q(f"PRAGMA table_info({TABLE_NAME})"))
row_count = list(db.q(f"SELECT COUNT(*) AS row_count FROM {TABLE_NAME}"))[0]["row_count"]

print(f"Table: {TABLE_NAME}")
print(f"Rows: {row_count}")
to_table(columns)

Table: discovered_urls
Rows: 1377


,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,site_id,INTEGER,0,None,0
2,2,url,TEXT,0,None,0
3,3,kind,TEXT,0,None,0
4,4,discovered_at,TEXT,0,None,0


In [4]:
SAMPLE_LIMIT = 25

if TABLE_NAME == "discovered_urls":
    sample = list(
        db.q(
            """
            SELECT id, site_id, url, kind, discovered_at
            FROM discovered_urls
            ORDER BY discovered_at DESC
            LIMIT ?
            """,
            [SAMPLE_LIMIT],
        )
    )
else:
    sample = list(db.q(f"SELECT * FROM {TABLE_NAME} LIMIT ?", [SAMPLE_LIMIT]))

to_table(sample)

,id,site_id,url,kind,discovered_at
0,2182,2,https://connections/?docs=bsc%2Fece%2Fusing-ec...,html,2026-02-12T14:48:26.098665
1,2181,2,https://connections/?docs=bsc%2Fece%2Fusing-ec...,html,2026-02-12T14:48:15.645805
2,2180,2,https://connections/?docs=bsc%2Fece%2Fusing-ec...,html,2026-02-12T14:48:08.935549
3,2179,2,https://connections/?docs=bsc%2Fece%2Fusing-ec...,html,2026-02-12T14:48:00.090314
4,2178,2,https://connections/?docs=residential%2Fstart-...,html,2026-02-12T14:47:52.944157
5,2177,2,https://connections/?docs=residential%2Fdeposi...,html,2026-02-12T14:47:44.591465
6,2176,2,https://connections/?docs=residential%2Fstart-...,html,2026-02-12T14:47:37.305681
7,2175,2,https://connections/?docs=residential%2Fbillin...,html,2026-02-12T14:47:29.320224
8,2174,2,https://connections/?docs=residential%2Fstart-...,html,2026-02-12T14:47:22.127368
9,2173,2,https://connections/?docs=residential%2Fcustom...,html,2026-02-12T14:47:14.910085


In [5]:
# Most recent pages by last_scraped
RECENT_LIMIT = 50

recent_pages = list(
    db.q(
        """
        SELECT id, site_id, url, last_scraped, last_changed
        FROM pages
        ORDER BY last_scraped DESC
        LIMIT ?
        """,
        [RECENT_LIMIT],
    )
)

to_table(recent_pages)

,id,site_id,url,last_scraped,last_changed
0,743,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:23.680557,2026-02-12T20:21:23.680557
1,742,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:19.401396,2026-02-12T20:21:19.401396
2,741,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:15.174977,2026-02-12T20:21:15.174977
3,740,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:10.766843,2026-02-12T20:21:10.766843
4,739,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:06.320414,2026-02-12T20:21:06.320414
5,738,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:21:01.578945,2026-02-12T20:21:01.578945
6,737,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:20:56.020925,2026-02-12T20:20:56.020925
7,736,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:20:51.517268,2026-02-12T20:20:51.517268
8,735,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:20:47.052126,2026-02-12T20:20:47.052126
9,734,2,https://connections/?docs=residential%2Fstart-...,2026-02-12T20:20:42.507674,2026-02-12T20:20:42.507674
